<div align="center">

# YAPAY ZEKÂ DESTEKLİ İLAÇ TASARIMI

**Erzincan Binali Yıldırım Üniversitesi — Sürekli Eğitim Uygulama ve Araştırma Merkezi**

28–29 Kasım 2026 · ML tabanlı QSAR uygulamalı eğitimi

</div>

---

### Eğitim sorumlusu

**Dr. Öğr. Üyesi Ekrem Yaşar**
Erzincan Binali Yıldırım Üniversitesi, Tıp Fakültesi, Biyofizik Anabilim Dalı

📧 `ey.biophysics@gmail.com`

---

# Oturum 1 — Veriyi Hazırlamak

Bu defterde, NTRK1 proteinine etki eden moleküllerin deneysel ölçümlerini
ChEMBL veritabanından alıp model eğitmeye hazır temiz bir tabloya çeviriyoruz.

**Neden ilk oturum veriye ayrıldı?** Model, verdiğimiz veriden ne öğrenebilirse
onu öğrenir. Veri kirliyse model de kirli olur.

### Bu defterde ne yapacağız?

1. Hedef proteinin ChEMBL'deki ölçümlerini **sayacağız** — hangi tipten kaç kayıt var?
2. En çok veriye sahip tipi (IC50) **ham hâliyle** indireceğiz.
3. Ham kayıtları **adım adım** temizleyeceğiz. Her adım ayrı bir hücrede: ne
   elendi, neden elendi.
4. IC50'yi `pIC50` ölçeğine çevireceğiz.
5. Dağılıma bakıp **sınıflandırma eşiğini** birlikte seçeceğiz.
6. İki veri setini Drive'a kaydedeceğiz.

### Sonunda ne üretilecek?

| Dosya | İçeriği | Kullanan oturum |
|---|---|---|
| `ntrk1_regresyon.csv` | `Smiles` + `pIC50` | Oturum 3 |
| `ntrk1_siniflandirma.csv` | `Smiles` + `pIC50` + `Aktiflik` (0/1) | Oturum 4 |

⏱️ Yaklaşık süre: 20–30 dakika

---
## Hedef protein: NTRK1 (TrkA)

| | |
|---|---|
| **Gen** | `NTRK1` |
| **UniProt** | [`P04629`](https://www.uniprot.org/uniprotkb/P04629) |
| **ChEMBL hedef** | [`CHEMBL2815`](https://www.ebi.ac.uk/chembl/target_report_card/CHEMBL2815/) |
| **Protein sınıfı** | Reseptör tirozin kinaz |
| **Doğal ligandı** | Sinir büyüme faktörü (NGF) |

**Neden bu hedef seçildi?**

- **Klinik olarak doğrulanmış.** `NTRK` gen füzyonlarını hedefleyen
  **larotrektinib** ve **entrektinib**, FDA tarihinde ilk kez tümörün bulunduğu
  dokuya değil taşıdığı genetik değişime göre onaylanan ilaçlardır.
- **Veri açısından zengin.** ChEMBL'de bu hedef için 10.000'in üzerinde IC50
  kaydı vardır. Bir modelin öğrenmesi için yeterli büyüklükte.
- **Direnç sorunu gerçek.** Birinci nesil inhibitörlere karşı gelişen direnç
  mutasyonları ikinci nesil ilaçları gerektirdi. Aktiflik tahmini, sürekli
  yenilenen bir problem.

---
## 1. Ayarlar

Defterdeki bütün sabitler bu tek hücrede. Başka bir hedefle çalışmak isterseniz
yalnızca burayı değiştirmeniz yeterli.

**`VERI_KAYNAGI`** veriyi nereden alacağımızı belirler:

| Değer | Ne yapar? |
|---|---|
| `"ikisi"` | **Varsayılan.** Canlı ChEMBL'den indirir, sonucu depodaki dondurulmuş sürümle karşılaştırır. |
| `"chembl"` | Yalnızca canlı ChEMBL. |
| `"depo_ham"` | ChEMBL'e **hiç bağlanmadan**, depodaki ham dosyadan çalışır. |

> **ChEMBL yanıt vermiyorsa** `"depo_ham"` seçin. Temizlik adımlarının hepsi
> yine çalışır ve birebir aynı sonucu verir.

**Sınıflandırma eşiğini burada görmüyorsunuz** — onu Bölüm 13'te, dağılıma
baktıktan sonra seçeceğiz.

In [ ]:
# ── Hedef ─────────────────────────────────────────────────────────────────────
TARGET_CHEMBL_ID = "CHEMBL2815"      # NTRK1 / TrkA  (UniProt: P04629)
TARGET_ADI       = "NTRK1"
AKTIVITE_TIPI    = "IC50"            # Modellenecek olcum tipi
HEDEF_KOLON      = "pIC50"           # -log10(IC50 [M])

# ── Mantikli pIC50 araligi ───────────────────────────────────────────────────
# Alt sinir 3.0 -> IC50 = 1 mM. Bunun disi veri giris hatasidir (bkz. Adim 9).
MANTIK_ARALIGI = (3.0, 12.0)

# ── Veri kaynagi:  "ikisi" | "chembl" | "depo_ham" ───────────────────────────
VERI_KAYNAGI = "ikisi"

# ── Google Drive proje klasoru ───────────────────────────────────────────────
# Butun oturumlar (01-06) bu ANA klasoru paylasir; her oturum kendi alt
# klasorune yazar.
DRIVE_PROJE_KLASORU = "/content/drive/MyDrive/EBYUSEM2026_MLQSAR"
YEREL_KLASOR        = "EBYUSEM2026_MLQSAR"      # Colab disi ortam icin

# ── Kurs deposu ──────────────────────────────────────────────────────────────
DEPO_RAW = "https://raw.githubusercontent.com/eygpcr/ebyusem2026-mlqsar/main"

# ── Dosya adlari ─────────────────────────────────────────────────────────────
AD_HAM = "ntrk1_ham_ic50.csv"
AD_REG = "ntrk1_regresyon.csv"
AD_SNF = "ntrk1_siniflandirma.csv"

# Depodaki ham ChEMBL disa aktarimlari (cevrimdisi yedek)
AD_DISA_IC50 = "chembl_disa_aktarim/ntrk1_chembl_ic50.csv.zip"
AD_DISA_TUM  = "chembl_disa_aktarim/ntrk1_chembl_tum_aktiviteler.csv.zip"

print(f"Hedef         : {TARGET_ADI} ({TARGET_CHEMBL_ID})")
print(f"Olcum tipi    : {AKTIVITE_TIPI}  ->  {HEDEF_KOLON}")
print(f"Veri kaynagi  : {VERI_KAYNAGI}")

---
## 2. Google Drive'ı bağlamak

Colab oturumları geçicidir: çalışma zamanı kapandığında `/content/` altındaki
her şey silinir. Çıktılarımız kalıcı olsun ve sonraki oturumlar onlara
erişebilsin diye **Drive**'a yazıyoruz.

Hücre çalışınca Colab izin isteyecek; açılan pencereden hesabınızı seçip onay
verin.

### Klasör düzeni

Her oturum **kendi alt klasörüne** yazar. Böylece dosyalar karışmaz ve hangi
çıktının nereden geldiği belli olur:

```
EBYUSEM2026_MLQSAR/
├── 01_veri/            ← bu defterin çıktıları
├── 02_ozellikler/
├── 03_regresyon/
├── 04_siniflandirma/
├── 05_analiz/
└── 06_adaylar/
```

Her klasörün içinde bir de `gorseller/` klasörü var; grafikler oraya kaydedilir.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJE, ORTAM = Path(DRIVE_PROJE_KLASORU), "Colab + Drive"
except (ImportError, ModuleNotFoundError):
    PROJE, ORTAM = Path(YEREL_KLASOR), "Yerel"

# Her oturumun kendi klasoru. Sonraki defterler de ayni sozlugu kurar,
# boylece bir oturum digerinin ciktisini adiyla bulur.
KLASOR = {
    1: PROJE / "01_veri",
    2: PROJE / "02_ozellikler",
    3: PROJE / "03_regresyon",
    4: PROJE / "04_siniflandirma",
    5: PROJE / "05_analiz",
    6: PROJE / "06_adaylar",
}
for k in KLASOR.values():
    (k / "gorseller").mkdir(parents=True, exist_ok=True)

BU     = KLASOR[1]             # bu defterin klasoru
GORSEL = BU / "gorseller"      # grafikler buraya

YOL_HAM = BU / AD_HAM
YOL_REG = BU / AD_REG
YOL_SNF = BU / AD_SNF

print(f"Ortam       : {ORTAM}")
print(f"Ana klasor  : {PROJE}")
print(f"Bu oturum   : {BU}")

# Eski duzende (dogrudan ana klasore) yazilmis dosyalar varsa haber ver.
eski = sorted(p.name for p in PROJE.glob("ntrk1_*")) + \
       sorted(p.name for p in PROJE.glob("*.joblib"))
if eski:
    print(f"\nNot: ana klasorde eski duzenden kalan {len(eski)} dosya var")
    print(f"     (ornek: {', '.join(eski[:3])}).")
    print("     Yeni ciktilar alt klasorlere yaziliyor; eskileri silebilirsiniz.")

---
## 3. Kütüphaneler

**`rdkit`** kemoinformatiğin standart açık kaynak kütüphanesi. SMILES
dizgilerini doğrulamak ve **kanonik** (tek, standart) biçime çevirmek için
kullanacağız. **`requests`** ile ChEMBL'in web servisine istek göndereceğiz.

Kurulum oturum başına bir kez gerekir.

In [ ]:
!pip install rdkit -q

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw

RDLogger.DisableLog("rdApp.*")        # RDKit'in ayristirma uyarilarini sustur

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 90)
plt.rcParams["figure.dpi"] = 110

print("Kutuphaneler yuklendi.")

---
## 4. ChEMBL'e sorgu atmak

ChEMBL web servisi büyük sorgularda zaman zaman geçici `500` hataları döndürür:
aynı istek bir denemede başarısız olurken bir sonrakinde çalışır.

Çözüm basit: isteği **artan bekleme süreleriyle yeniden denemek**. Aşağıdaki
`chembl_sorgu` bunu yapar — başarısız olursa bekler, bekleme süresini katlar,
tekrar dener.

İki fonksiyon daha var:

- `chembl_kayit_sayisi` — veriyi indirmeden yalnızca **sayar**. Saniyeler sürer.
- `chembl_aktiviteleri_indir` — kayıtları **sayfa sayfa** indirir (ChEMBL bir
  istekte en çok 1000 kayıt verir).

### Çevrimdışı yedek

ChEMBL tamamen kapalıysa diye kurs deposunda, ChEMBL'in web arayüzünden
indirilmiş ham dosyalar duruyor. `disa_aktarim_indir()` bu dosyayı Drive
klasörünüze indirir ve **zaten varsa yeniden indirmez**. Dosya bir kez iner,
Colab oturumu kapansa bile Drive'da kalır.

Tek fark kolon adlarıdır: web dışa aktarımı `Molecule ChEMBL ID` gibi
insan-okunur başlıklar kullanır, REST servisi `molecule_chembl_id` gibi alan
adları. `disa_aktarim_oku()` bu çevirmeyi yapar, böylece temizlik adımları iki
kaynak için de **aynı kodla** çalışır.

In [ ]:
CHEMBL_API = "https://www.ebi.ac.uk/chembl/api/data"


def chembl_sorgu(uc_nokta: str, parametreler: dict, deneme: int = 15) -> dict:
    """ChEMBL istegi; gecici hatalara karsi artan beklemeyle yeniden dener."""
    url = f"{CHEMBL_API}/{uc_nokta}.json"
    bekleme, son_hata = 1.0, None
    for k in range(deneme):
        try:
            yanit = requests.get(url, params=parametreler, timeout=120)
            yanit.raise_for_status()
            return yanit.json()
        except Exception as hata:
            son_hata = hata
            if k < deneme - 1:
                time.sleep(bekleme)
                bekleme = min(bekleme * 1.8, 30)   # 1s, 1.8s, 3.2s ... en cok 30s
    raise RuntimeError(f"ChEMBL {deneme} denemede yanit vermedi: {son_hata}")


def chembl_kayit_sayisi(target_id: str, tip: str, sadece_kesin: bool) -> int:
    """Kayit sayisini verir; veriyi indirmez (tek hafif istek)."""
    p = {"target_chembl_id": target_id, "standard_type": tip, "limit": 1}
    if sadece_kesin:
        p["standard_relation"] = "="     # yalnizca kesin olcumler
        p["standard_units"]    = "nM"    # birim tutarliligi
    return chembl_sorgu("activity", p)["page_meta"]["total_count"]


def chembl_aktiviteleri_indir(target_id: str, tip: str) -> pd.DataFrame:
    """Hedefin verilen tipteki BUTUN kayitlarini indirir — hic suzmeden.

    Suzme islerini bilerek burada yapmiyoruz: her suzgeci Bolum 8'de ayri bir
    adim olarak, ne elendigini gorerek uygulayacagiz.
    """
    kayitlar, offset = [], 0
    while True:
        yanit = chembl_sorgu("activity", {
            "target_chembl_id": target_id,
            "standard_type": tip,
            "limit": 1000,
            "offset": offset,
            "only": ("molecule_chembl_id,canonical_smiles,standard_type,"
                     "standard_relation,standard_value,standard_units,"
                     "assay_chembl_id,pchembl_value"),
        })
        sayfa = yanit.get("activities", [])
        kayitlar.extend(sayfa)
        print(f"  indirildi: {len(kayitlar):>6} / {yanit['page_meta']['total_count']}")
        if len(sayfa) < 1000:
            break
        offset += 1000
    return pd.DataFrame(kayitlar)


# ── Cevrimdisi yedek ─────────────────────────────────────────────────────────

DISA_AKTARIM_ESLEME = {
    "Molecule ChEMBL ID": "molecule_chembl_id",
    "Smiles":              "canonical_smiles",
    "Standard Type":       "standard_type",
    "Standard Relation":   "standard_relation",
    "Standard Value":      "standard_value",
    "Standard Units":      "standard_units",
    "Assay ChEMBL ID":     "assay_chembl_id",
    "pChEMBL Value":       "pchembl_value",
}


def disa_aktarim_indir(dosya_adi: str):
    """Depodaki ham dosyayi Drive klasorune indirir; zaten varsa indirmez."""
    hedef = BU / Path(dosya_adi).name
    if hedef.exists():
        print(f"  Drive'da zaten var : {hedef.name}  "
              f"({hedef.stat().st_size / 1024 / 1024:.1f} MB)")
        return hedef

    url = f"{DEPO_RAW}/03_veri/{dosya_adi}"
    print(f"  indiriliyor        : {dosya_adi}")
    yanit = requests.get(url, timeout=300, stream=True)
    yanit.raise_for_status()
    with open(hedef, "wb") as dosya:
        for parca in yanit.iter_content(chunk_size=1 << 20):
            dosya.write(parca)
    print(f"  Drive'a kaydedildi : {hedef.name}  "
          f"({hedef.stat().st_size / 1024 / 1024:.1f} MB)")
    return hedef


def disa_aktarim_oku(zip_yolu) -> pd.DataFrame:
    """Zip'li disa aktarimi okur ve kolon adlarini REST adlarina cevirir.

    Zip ACILMAZ: pandas sikistirilmis CSV'yi dogrudan okur.
    """
    tablo = pd.read_csv(zip_yolu, sep=";", compression="zip", low_memory=False)
    eksik = [k for k in DISA_AKTARIM_ESLEME if k not in tablo.columns]
    if eksik:
        raise KeyError(f"Disa aktarimda beklenen kolonlar yok: {eksik}")
    ham = tablo[list(DISA_AKTARIM_ESLEME)].rename(columns=DISA_AKTARIM_ESLEME)
    # Disa aktarimda iliski bazen "'='" bicimindedir; tirnaklari temizliyoruz.
    ham["standard_relation"] = (ham["standard_relation"].astype(str).str.strip()
                                .str.replace("'", "", regex=False)
                                .str.replace('"', "", regex=False))
    return ham


print("Sorgu fonksiyonlari hazir (canli servis + cevrimdisi yedek).")

---
## 5. Hedefte hangi ölçümden kaç kayıt var?

Modellemeye başlamadan önce elimizde **ne olduğunu** görmek gerekir. Bir QSAR
modeli **tek ve tutarlı bir ölçüm tipi** üzerine kurulmalıdır; farklı deney
tiplerini aynı torbaya koymak modele gürültü katar.

| Ölçüm | Ne ölçer? |
|---|---|
| **IC50** | Hedefin aktivitesini %50 baskılamak için gereken derişim |
| **Ki** | Bağlanma afinitesinin denge sabiti |
| **Kd** | Ayrışma sabiti (doğrudan bağlanma) |
| **EC50** | %50 etki oluşturan derişim |
| **Potency** | Deneye özgü potens ölçümü |
| **Inhibition** | Tek derişimde yüzde baskılama |

Aşağıdaki hücre her tip için **iki** sayı verir:

- **Ham kayıt** — hiçbir süzgeç uygulanmadan, o tipteki bütün kayıtlar.
- **Kesin (=) + nM** — ölçümün kesin olduğu (`>` veya `<` değil) ve biriminin
  `nM` olduğu kayıtlar.

Aradaki fark önemlidir: ham sayı büyük görünebilir ama modellenebilir kısım
daha küçüktür. Veriyi indirmiyoruz, yalnızca sayıyoruz.

In [ ]:
TIPLER = ["IC50", "Ki", "Kd", "EC50", "Potency", "Inhibition"]

sayim_tablo, SAYIM_KAYNAK = None, None

# --- Yol A: canli servisten say ---
if VERI_KAYNAGI != "depo_ham":
    satirlar = []
    for tip in TIPLER:
        # Her tip BAGIMSIZ denenir: biri basarisiz olursa digerleri yine sayilir.
        try:
            ham_n   = chembl_kayit_sayisi(TARGET_CHEMBL_ID, tip, False)
            kesin_n = chembl_kayit_sayisi(TARGET_CHEMBL_ID, tip, True)
            satirlar.append({"Olcum tipi": tip, "Ham kayit": ham_n,
                             "Kesin (=) + nM": kesin_n})
            print(f"  {tip:<11} ham {ham_n:>7}    kesin+nM {kesin_n:>7}")
        except Exception:
            print(f"  {tip:<11} sayilamadi (ChEMBL yanit vermedi)")
    if satirlar:
        sayim_tablo = pd.DataFrame(satirlar)
        SAYIM_KAYNAK = "canli ChEMBL servisi"

# --- Yol B: depodaki "butun aktiviteler" dosyasindan say ---
if sayim_tablo is None:
    print("Depodaki butun-aktiviteler dosyasindan sayiliyor...")
    tum = disa_aktarim_oku(disa_aktarim_indir(AD_DISA_TUM))
    tip_kolon = tum["standard_type"].astype(str).str.strip()
    kesin = tum[(tum["standard_relation"] == "=")
                & (tum["standard_units"].astype(str).str.strip() == "nM")]
    sayim_tablo = (
        pd.DataFrame({
            "Ham kayit": tip_kolon.value_counts(),
            "Kesin (=) + nM": kesin["standard_type"].astype(str).str.strip()
                              .value_counts(),
        })
        .fillna(0).astype(int)
        .sort_values("Ham kayit", ascending=False)
        .head(10)
        .rename_axis("Olcum tipi").reset_index()
    )
    SAYIM_KAYNAK = "depo (butun aktiviteler disa aktarimi)"

sayim_tablo["Kullanilabilir %"] = (
    100 * sayim_tablo["Kesin (=) + nM"]
    / sayim_tablo["Ham kayit"].replace(0, np.nan)).round(1)

print()
print(f"Kaynak: {SAYIM_KAYNAK}")
print("=" * 58)
print(sayim_tablo.to_string(index=False))

en_cok = sayim_tablo.sort_values("Kesin (=) + nM", ascending=False).iloc[0]
print(f"\nEn cok kullanilabilir veri '{en_cok['Olcum tipi']}' tipinde: "
      f"{en_cok['Kesin (=) + nM']} kayit.")
print(f"Modellemeyi bu yuzden {AKTIVITE_TIPI} uzerine kuracagiz.")

### Aynı tabloyu grafikle görmek

Sayılar arasındaki fark çok büyük olduğu için yatay eksen **logaritmik**.
Her çubuk çiftinde soldaki ham kayıt sayısı, sağdaki kullanılabilir kısım.

In [ ]:
cizim = sayim_tablo.sort_values("Ham kayit")
y = np.arange(len(cizim))
yuk = 0.38

fig, eksen = plt.subplots(figsize=(10, 0.62 * len(cizim) + 1.8))

eksen.barh(y + yuk / 2, cizim["Ham kayit"], height=yuk,
           color="#bdc3c7", edgecolor="white", label="Ham kayit")
eksen.barh(y - yuk / 2, cizim["Kesin (=) + nM"], height=yuk,
           color="#2980b9", edgecolor="white", label="Kesin (=) + nM")

for i, satir in enumerate(cizim.itertuples()):
    eksen.text(satir._2 * 1.08, i + yuk / 2, f"{satir._2:,}",
               va="center", fontsize=8.5, color="#555")
    if satir._3 > 0:
        eksen.text(satir._3 * 1.08, i - yuk / 2, f"{satir._3:,}",
                   va="center", fontsize=8.5, color="#2980b9")

eksen.set_yticks(y, cizim["Olcum tipi"])
eksen.set_xscale("log")
eksen.set_xlim(right=cizim["Ham kayit"].max() * 4)
eksen.set_xlabel("Kayit sayisi (logaritmik olcek)")
eksen.set_title(f"{TARGET_ADI} — olcum tiplerine gore kayit sayisi")
eksen.legend(loc="lower right")
eksen.grid(alpha=0.3, axis="x", linestyle="--")

plt.tight_layout()
plt.savefig(GORSEL / "01_olcum_tipleri.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Ham IC50 kayıtlarını indirmek

Şimdi IC50 kayıtlarının **hepsini** indiriyoruz — hiçbir süzgeç uygulamadan.
Süzme işlerini bilerek burada yapmıyoruz: her süzgeci bir sonraki bölümde ayrı
bir adım olarak, **ne elendiğini görerek** uygulayacağız.

İndirme bittiğinde ham veri hemen Drive'a kaydedilir; hücreyi tekrar
çalıştırmak zorunda kalmazsınız.

> ⏱️ Birkaç dakika sürebilir. ChEMBL yanıt vermezse kod kendiliğinden depodaki
> ham dosyaya düşer ve aynı noktadan devam eder.

In [ ]:
ham, HAM_KAYNAK = None, None

# --- Yol A: canli servis ---
if VERI_KAYNAGI in ("ikisi", "chembl"):
    print(f"'{TARGET_CHEMBL_ID}' icin butun {AKTIVITE_TIPI} kayitlari indiriliyor...")
    try:
        ham = chembl_aktiviteleri_indir(TARGET_CHEMBL_ID, AKTIVITE_TIPI)
        HAM_KAYNAK = "canli ChEMBL servisi"
    except Exception as hata:
        print(f"\n!! Canli cekim basarisiz oldu: {hata}")
        print("   Depodaki ham dosyaya dusuluyor.")

# --- Yol B: depodaki ham dosya ---
if ham is None:
    yol_disa = disa_aktarim_indir(AD_DISA_IC50)
    disa = disa_aktarim_oku(yol_disa)
    ham = disa[disa["standard_type"].astype(str).str.strip() == AKTIVITE_TIPI].copy()
    HAM_KAYNAK = "depo (ham ChEMBL disa aktarimi)"

ham = ham.reset_index(drop=True)
ham.to_csv(YOL_HAM, index=False)

print()
print(f"Ham kayit sayisi : {len(ham)}")
print(f"Kaynak           : {HAM_KAYNAK}")
print(f"Kaydedildi       : {YOL_HAM.name}")
ham.head()

---
## 7. Temizlik — adım adım

Ham deneysel veri her zaman kirlidir: sansürlü ölçümler, farklı birimler, eksik
alanlar, aynı molekülün birden çok kaydı, geçersiz yapılar.

Aşağıda **dokuz adım** var. Her adım ayrı bir hücrede ve her hücre şunu söyler:
kaç kayıt kaldı, kaç kayıt elendi, **neden** elendi.

| Adım | Ne yapar? |
|---|---|
| 1 | Sansürlü ölçümleri (`>`, `<`) atar |
| 2 | Birimi `nM` olmayanları atar |
| 3 | Eksik alanı olan satırları atar |
| 4 | Değeri sayıya çevirir, pozitif olanları tutar |
| 5 | Molekül başına tek ölçüm seçer |
| 6 | SMILES'ı kanonikleştirir, geçersiz yapıları atar |
| 7 | Kanonik yapıya göre yeniden tekilleştirir |
| 8 | `pIC50` hesaplar |
| 9 | Fiziksel olarak imkânsız değerleri atar |

Her adım, bir öncekinin çıktısından **yeni bir tablo** üretir (`d1`, `d2`, ...).
Böylece bir hücreyi iki kez çalıştırsanız bile sonuç değişmez ve istediğiniz
adıma geri dönüp bakabilirsiniz.

Önce tek bir yardımcı fonksiyon tanımlayalım.

In [ ]:
def kanonik_smiles(smiles):
    """SMILES'i kanonik RDKit gosterimine cevirir; gecersiz/eksikse None."""
    if not isinstance(smiles, str) or not smiles:
        return None
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol, canonical=True) if mol else None


def adim_raporu(etiket, once, sonra):
    """Adimin ne kadar veri eledigini tek satirda yazdirir."""
    elenen = once - sonra
    print(f"{etiket}")
    print(f"  once  : {once:>7}")
    print(f"  elenen: {elenen:>7}   (%{100 * elenen / once:.1f})" if once else "")
    print(f"  kalan : {sonra:>7}")


print("Yardimci fonksiyonlar hazir.")

### Adım 1 — Sansürlü ölçümleri atmak

ChEMBL'de bir ölçüm `=` yerine `>` veya `<` ilişkisiyle kaydedilmiş olabilir.
`IC50 > 10000 nM` demek, **gerçek değerin bilinmediği** anlamına gelir: deney
en yüksek derişimde bile %50 baskılama görmemiştir.

Böyle bir kaydı regresyona gerçek sayı gibi vermek modeli yanıltır — "bu
molekülün IC50'si tam 10000 nM" demiş oluruz, oysa 50000 de olabilir.

> İleri not: bu kayıtları atmak yerine modelleyen yöntemler vardır
> (*censored regression*, *survival analysis*). Bu kursun kapsamı dışında.

In [ ]:
iliski = ham["standard_relation"].astype(str).str.strip()

print("Iliski tiplerinin dagilimi")
print("-" * 34)
print(iliski.value_counts().to_string())

d1 = ham[iliski == "="].copy()

print()
adim_raporu("Adim 1 — kesin (=) olcumler", len(ham), len(d1))

atilan = ham[iliski.isin([">", "<"])]
if len(atilan):
    print("\nAtilan kayitlara ornek:")
    print(atilan[["molecule_chembl_id", "standard_relation",
                  "standard_value", "standard_units"]].head(3).to_string(index=False))

### Adım 2 — Birim tutarlılığı

`pIC50` hesabı **molar derişim** gerektirir. ChEMBL'de değerler bazen
`µg.mL⁻¹` gibi birimlerde gelir; bunları molar derişime çevirmek için molekül
kütlesi gerekir ve her kayıtta güvenilir biçimde bulunmaz.

Bu yüzden yalnızca `nM` birimli kayıtlarla ilerliyoruz.

In [ ]:
birim = d1["standard_units"].astype(str).str.strip()

print("Birimlerin dagilimi")
print("-" * 30)
print(birim.value_counts().head(8).to_string())

d2 = d1[birim == "nM"].copy()

print()
adim_raporu("Adim 2 — birim nM", len(d1), len(d2))

### Adım 3 — Eksik alanlar

Bir kaydı modelleyebilmek için üç şey gerekir: molekülün kimliği, yapısı
(`SMILES`) ve ölçülen değeri. Herhangi biri eksikse kayıt kullanılamaz.

In [ ]:
gerekli = ["molecule_chembl_id", "standard_value", "canonical_smiles"]

print("Eksik deger sayilari")
print("-" * 36)
print(d2[gerekli].isna().sum().to_string())

d3 = d2.dropna(subset=gerekli).reset_index(drop=True)

print()
adim_raporu("Adim 3 — butun alanlar tam", len(d2), len(d3))

### Adım 4 — Değeri sayıya çevirmek

Değerler CSV'den metin olarak gelebilir. Sayıya çeviriyoruz; çevrilemeyenler
`NaN` olur ve atılır.

Ayrıca değerin **pozitif** olması zorunlu: bir sonraki adımlarda logaritma
alacağız, sıfırın veya negatifin logaritması yok.

In [ ]:
d4 = d3.copy()
d4["standard_value"] = pd.to_numeric(d4["standard_value"], errors="coerce")

sayiya_cevrilemeyen = int(d4["standard_value"].isna().sum())
d4 = d4.dropna(subset=["standard_value"])

pozitif_olmayan = int((d4["standard_value"] <= 0).sum())
d4 = d4[d4["standard_value"] > 0].reset_index(drop=True)

print(f"Sayiya cevrilemeyen : {sayiya_cevrilemeyen}")
print(f"Sifir veya negatif  : {pozitif_olmayan}")
print()
adim_raporu("Adim 4 — sayisal ve pozitif", len(d3), len(d4))

print(f"\n{AKTIVITE_TIPI} (nM) araligi: {d4['standard_value'].min():,.3f} - "
      f"{d4['standard_value'].max():,.0f}")

### Adım 5 — Molekül başına tek ölçüm

Aynı molekül farklı laboratuvarlarda, farklı deneylerde ölçülmüş olabilir.
Modelin her molekül için **tek** bir hedef değere ihtiyacı var.

Biz **en düşük** IC50'yi, yani **en potent** ölçümü seçiyoruz. Gerekçesi: bir
molekülün gerçek potensi, onu en uygun koşulda ölçen deneyde görünür; zayıf
sonuçlar genellikle deney koşullarının elverişsizliğinden gelir.

> Alternatif, ölçümlerin **medyanını** almaktır. Aykırı bir ölçüme karşı daha
> dayanıklıdır ama potent uçları yumuşatır. Hangisini seçerseniz seçin,
> **bildirin**.

In [ ]:
tekrar = d4["molecule_chembl_id"].value_counts()

print(f"Benzersiz molekul       : {len(tekrar)}")
print(f"Birden cok olcumu olan  : {int((tekrar > 1).sum())}")
print(f"En cok olcumu olan      : {tekrar.index[0]} ({tekrar.iloc[0]} olcum)")

# Ornek: ayni molekulun farkli olcumleri ne kadar dagilik?
ornek_id = tekrar.index[0]
ornek = d4[d4["molecule_chembl_id"] == ornek_id]["standard_value"]
print(f"\n{ornek_id} icin olculen degerler (nM):")
print(f"  en kucuk {ornek.min():,.1f}   medyan {ornek.median():,.1f}   "
      f"en buyuk {ornek.max():,.1f}")

d5 = (d4.loc[d4.groupby("molecule_chembl_id")["standard_value"].idxmin()]
      .reset_index(drop=True))

print()
adim_raporu("Adim 5 — molekul basina en potent olcum", len(d4), len(d5))

### Adım 6 — SMILES'ı kanonikleştirmek

Aynı molekül birden çok biçimde yazılabilir: `OCC` ve `CCO` ikisi de etanoldur.
RDKit'in **kanonikleştirmesi** her yapı için tek bir standart yazım üretir.

Bu adım aynı zamanda bir **geçerlilik kontrolüdür**: RDKit okuyamadığı bir
SMILES için `None` döner, biz de o kaydı atarız.

**Neden kritik?** Farklı yazımlar yüzünden aynı molekül veri setinde iki ayrı
satır gibi görünür. Bu "gizli tekrarlar" yalnızca veriyi şişirmez; eğitim/test
bölünmesinde **aynı molekülün hem eğitimde hem testte** yer almasına yol açar.
Buna **veri sızıntısı** denir ve modelin başarısını olduğundan iyi gösterir.

In [ ]:
d6 = d5.copy()
d6["Smiles"] = d6["canonical_smiles"].apply(kanonik_smiles)

gecersiz = int(d6["Smiles"].isna().sum())
d6 = d6.dropna(subset=["Smiles"]).reset_index(drop=True)

print(f"RDKit'in okuyamadigi SMILES : {gecersiz}")
print()
adim_raporu("Adim 6 — gecerli kanonik SMILES", len(d5), len(d6))

# Kanoniklestirmenin ne degistirdigine bir ornek
degisen = d6[d6["canonical_smiles"] != d6["Smiles"]]
print(f"\nYazimi degisen molekul: {len(degisen)} / {len(d6)}")
if len(degisen):
    s = degisen.iloc[0]
    print(f"  ChEMBL'deki : {s['canonical_smiles']}")
    print(f"  Kanonik     : {s['Smiles']}")

### Adım 7 — Gizli tekrarları temizlemek

Adım 5'te **ChEMBL kimliğine** göre tekilleştirdik. Ama aynı yapı ChEMBL'de iki
farklı kimlikle duruyor olabilir (ör. farklı tuz biçimleri, farklı kayıtlar).
Kanonikleştirmeden **sonra** bunlar artık aynı dizgiye sahip olur ve
yakalanabilir.

Bu yüzden şimdi bir kez daha, **yapıya** göre tekilleştiriyoruz.

In [ ]:
yapi_tekrari = d6["Smiles"].duplicated().sum()
print(f"Ayni yapiyi tasiyan fazladan kayit: {int(yapi_tekrari)}")

if yapi_tekrari:
    ornek_yapi = d6[d6["Smiles"].duplicated(keep=False)].sort_values("Smiles")
    print("\nOrnek (ayni yapi, farkli ChEMBL kimligi):")
    print(ornek_yapi[["molecule_chembl_id", "standard_value", "Smiles"]]
          .head(4).to_string(index=False))

d7 = (d6.loc[d6.groupby("Smiles")["standard_value"].idxmin()]
      .reset_index(drop=True))

print()
adim_raporu("Adim 7 — yapiya gore benzersiz", len(d6), len(d7))

### Adım 8 — `pIC50` hesaplamak

IC50 değerleri çok geniş bir aralığa yayılır: 0,1 nM ile 1.000.000 nM arası.
Bu ölçek bir modelin öğrenmesi için elverişsizdir.

Çözüm, logaritmik dönüşüm:

$$\mathrm{pIC50} = -\log_{10}\left(\mathrm{IC50}\ [\mathrm{M}]\right)
= 9 - \log_{10}\left(\mathrm{IC50}\ [\mathrm{nM}]\right)$$

Dönüşüm iki işi birden yapar:

- **Yönü sezgisel hâle getirir.** Düşük IC50 = güçlü molekül. Logaritmadan
  sonra **yüksek pIC50 = güçlü molekül**.
- **Ölçeği sıkıştırır.** Altı büyüklük mertebesi, 3–10 arası dar bir aralığa iner.

| pIC50 | IC50 | Yorum |
|---|---|---|
| 9 | 1 nM | Çok potent |
| 8 | 10 nM | Potent |
| 7 | 100 nM | İyi |
| 6 | 1 µM | Orta |
| 5 | 10 µM | Zayıf |

In [ ]:
d8 = d7.copy()
d8[HEDEF_KOLON] = -np.log10(d8["standard_value"] * 1e-9)

print(f"{HEDEF_KOLON} hesaplandi.")
print()
print("Ornek cevrimler")
print("-" * 44)
for _, s in d8.sort_values("standard_value").iloc[[0, len(d8) // 2, -1]].iterrows():
    print(f"  {s['standard_value']:>18,.2f} nM  ->  {HEDEF_KOLON} = "
          f"{s[HEDEF_KOLON]:.2f}")

# ChEMBL kendi pChEMBL degerini de veriyor: hesabimiz dogru mu?
karsilastir = d8.dropna(subset=["pchembl_value"]).copy()
if len(karsilastir):
    karsilastir["pchembl_value"] = pd.to_numeric(karsilastir["pchembl_value"],
                                                 errors="coerce")
    karsilastir = karsilastir.dropna(subset=["pchembl_value"])
    fark = (karsilastir[HEDEF_KOLON] - karsilastir["pchembl_value"]).abs()
    print(f"\nChEMBL'in pChEMBL degeriyle karsilastirma ({len(karsilastir)} kayit)")
    print(f"  ortalama |fark| : {fark.mean():.4f}")
    print(f"  en buyuk |fark| : {fark.max():.4f}")
    print("  -> Hesabimiz ChEMBL ile tutarli." if fark.max() < 0.02
          else "  -> Fark var; yuvarlama disinda bir sebep olabilir.")

### Adım 9 — Fiziksel olarak imkânsız değerler

Bir değer sonsuz olmadığı hâlde imkânsız olabilir. Bu hedefte IC50'si
**9,8 molar** girilmiş bir kayıt var — böyle bir derişim deneysel olarak
uygulanamaz, açık bir veri giriş hatasıdır.

Bu yüzden mantıklı bir aralık koyuyoruz:

- **Alt sınır `pIC50 = 3`** → IC50 = 1 mM. Daha zayıf bir "aktivite" anlamsızdır.
- **Üst sınır `pIC50 = 12`** → IC50 = 1 pM. Bundan güçlü bir küçük molekül
  pratikte yok.

> Bu adım, hazır bir veri setini gözü kapalı kullanmanın neden riskli olduğunu
> gösterir. Tek bir uç değer, regresyon modelinin hata ölçütlerini tek başına
> bozabilir.

In [ ]:
alt, ust = MANTIK_ARALIGI

disarida = d8[(d8[HEDEF_KOLON] < alt) | (d8[HEDEF_KOLON] > ust)]
if len(disarida):
    print(f"Araligin ({alt}-{ust}) disinda kalan kayitlar:")
    print(disarida[["molecule_chembl_id", "standard_value", HEDEF_KOLON]]
          .to_string(index=False))
    en_uc = disarida.iloc[0]
    print(f"\n  {en_uc['molecule_chembl_id']}: {en_uc['standard_value']:,.0f} nM "
          f"= {en_uc['standard_value'] * 1e-9:,.1f} MOLAR")
else:
    print(f"Araligin ({alt}-{ust}) disinda kayit yok.")

d9 = d8[(d8[HEDEF_KOLON] >= alt) & (d8[HEDEF_KOLON] <= ust)].reset_index(drop=True)

print()
adim_raporu(f"Adim 9 — mantikli aralik {alt}-{ust}", len(d8), len(d9))

---
## 8. Temizlik hunisi

Dokuz adımı bitirdik. Şimdi geriye dönüp **ne kadar veri kaybettiğimizi** tek
tabloda görelim.

Bu tablo iki açıdan değerlidir:

- **Bilimsel raporlama.** Bir makalenin yöntem bölümündeki veri küratörlüğü
  adımlarını doğrudan bu tabloyla sunabilirsiniz.
- **Hata ayıklama.** Bir adımda beklenmedik büyüklükte kayıp olursa (örneğin
  verinin %90'ı gidiyorsa) sorun hemen görünür.

In [ ]:
df = (d9[["molecule_chembl_id", "Smiles", "standard_value", HEDEF_KOLON]]
      .rename(columns={"standard_value": f"{AKTIVITE_TIPI}_nM"})
      .sort_values(HEDEF_KOLON, ascending=False)
      .reset_index(drop=True))

huni = [
    ("Ham IC50 kaydi",                 len(ham)),
    ("1. Kesin olcum (=)",             len(d1)),
    ("2. Birim nM",                    len(d2)),
    ("3. Butun alanlar tam",           len(d3)),
    ("4. Sayisal ve pozitif",          len(d4)),
    ("5. Molekul basina tek olcum",    len(d5)),
    ("6. Gecerli kanonik SMILES",      len(d6)),
    ("7. Yapiya gore benzersiz",       len(d7)),
    ("8. pIC50 hesaplandi",            len(d8)),
    ("9. Mantikli aralik",             len(d9)),
]

print(f"TEMIZLIK HUNISI   (kaynak: {HAM_KAYNAK})")
print("=" * 62)
print(f"  {'Adim':<32}{'Kalan':>8}{'Elenen':>9}")
print("-" * 62)
onceki = None
for etiket, n in huni:
    elenen = "" if onceki is None else f"{onceki - n:>9,}"
    print(f"  {etiket:<32}{n:>8,}{elenen}")
    onceki = n
print("-" * 62)
print(f"  {'NIHAI VERI SETI':<32}{len(df):>8,}")
print(f"\nHam kayitlarin %{100 * len(df) / len(ham):.1f} kadari modellenebilir "
      f"molekule donustu.")
print(f"{HEDEF_KOLON} araligi: {df[HEDEF_KOLON].min():.2f} - {df[HEDEF_KOLON].max():.2f}")
df.head()

### Huniyi grafikle görmek

Çubukların boyu kalan kayıt sayısını, yanındaki gri kısım o adımda elenen
kayıtları gösterir.

In [ ]:
etiketler = [e for e, _ in huni]
degerler  = [n for _, n in huni]
y = np.arange(len(huni))[::-1]

fig, eksen = plt.subplots(figsize=(10.5, 0.55 * len(huni) + 1.6))

eksen.barh(y, degerler, color="#2980b9", edgecolor="white", height=0.68)
for i, (yer, n) in enumerate(zip(y, degerler)):
    elenen = 0 if i == 0 else degerler[i - 1] - n
    eksen.text(n + max(degerler) * 0.012, yer, f"{n:,}",
               va="center", fontsize=9, color="#2c3e50")
    if elenen:
        eksen.text(n + max(degerler) * 0.10, yer, f"(-{elenen:,})",
                   va="center", fontsize=8.5, color="#95a5a6")

eksen.set_yticks(y, etiketler, fontsize=9)
eksen.set_xlim(0, max(degerler) * 1.28)
eksen.set_xlabel("Kalan kayit sayisi")
eksen.set_title(f"{TARGET_ADI} {AKTIVITE_TIPI} — temizlik hunisi")
eksen.grid(alpha=0.3, axis="x", linestyle="--")

plt.tight_layout()
plt.savefig(GORSEL / "02_temizlik_hunisi.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 9. Sonucu depodaki sürümle karşılaştırmak

Kurs deposunda, **tam aynı** adımlardan geçirilmiş dondurulmuş bir sürüm var.
Onu üreten betiği de depoda bulabilirsiniz:
[`04_araclar/veri_uret.py`](https://github.com/eygpcr/ebyusem2026-mlqsar/blob/main/04_araclar/veri_uret.py).

**Neden karşılaştırıyoruz?** ChEMBL sürekli güncellenir. Aynı sorgu altı ay
sonra farklı sonuç verebilir: yeni ölçümler eklenir, hatalı kayıtlar geri
çekilir. Küçük farklar **normaldir ve beklenir**.

Bu yüzden yayınlarınızda şu üçünü mutlaka bildirin: ChEMBL **sürümü**, indirme
**tarihi** ve uyguladığınız **filtreler**.

> Bu bölüm yalnızca `VERI_KAYNAGI = "ikisi"` iken çalışır.

In [ ]:
df_depo = None

if VERI_KAYNAGI == "ikisi":
    url = f"{DEPO_RAW}/03_veri/{AD_REG}"
    try:
        df_depo = pd.read_csv(url)
        print(f"Depo surumu okundu: {len(df_depo)} molekul")
    except Exception as hata:
        print(f"!! Depo surumu okunamadi: {hata}")

if df_depo is not None:
    set_bizim = set(df["Smiles"])
    set_depo  = set(df_depo["Smiles"])
    ortak     = set_bizim & set_depo

    print()
    print("Molekul sayilari")
    print("-" * 46)
    print(f"  Bu calistirma      : {len(set_bizim):>6}")
    print(f"  Depo (dondurulmus) : {len(set_depo):>6}")
    print(f"  Ortak yapi         : {len(ortak):>6}")
    print(f"  Yalnizca bizde     : {len(set_bizim - set_depo):>6}")
    print(f"  Yalnizca depoda    : {len(set_depo - set_bizim):>6}")

    birlesik = df[["Smiles", HEDEF_KOLON]].merge(
        df_depo[["Smiles", HEDEF_KOLON]], on="Smiles", suffixes=("_bizim", "_depo"))
    fark = (birlesik[f"{HEDEF_KOLON}_bizim"]
            - birlesik[f"{HEDEF_KOLON}_depo"]).abs()

    print()
    print("Ortak molekullerde deger uyumu")
    print("-" * 46)
    print(f"  Karsilastirilan   : {len(birlesik):>6}")
    print(f"  Birebir ayni      : {int((fark < 1e-9).sum()):>6}"
          f"  (%{100 * (fark < 1e-9).mean():.1f})")
    print(f"  En buyuk |fark|   : {fark.max():.6f}")

    if fark.max() < 1e-9 and not (set_bizim ^ set_depo):
        print("\n=> Iki kaynak TAMAMEN ayni. Veri hatti tekrar uretilebilir.")
    else:
        print("\n=> Kucuk farklar var: ChEMBL depo tarihinden sonra guncellenmis.")
        print("   Beklenen bir durum; veri surumu bildirmenin neden zorunlu")
        print("   oldugunu gosterir.")
else:
    print("(Karsilastirma atlandi.)")

---
## 10. Veri setini tanımak

Modelleme öncesi her zaman veriye **bakılır**. Özet istatistikler, modelin
öğrenmesi beklenen sinyalin nasıl dağıldığını gösterir.

In [ ]:
print(f"{HEDEF_KOLON} ozet istatistikleri")
print("-" * 34)
print(df[HEDEF_KOLON].describe().round(3).to_string())

en_iyi, en_kotu = df[HEDEF_KOLON].max(), df[HEDEF_KOLON].min()
print(f"\nEn potent molekul : {HEDEF_KOLON} = {en_iyi:.2f}"
      f"   ({AKTIVITE_TIPI} ~ {10 ** (9 - en_iyi):.3f} nM)")
print(f"En zayif molekul  : {HEDEF_KOLON} = {en_kotu:.2f}"
      f"   ({AKTIVITE_TIPI} ~ {10 ** (9 - en_kotu):,.0f} nM)")

### En potent ve en zayıf moleküller

Sayılardan sonra **moleküllere bakmak** da önemlidir.

Bakarken şunu sorun: en potent moleküller ortak bir iskelet paylaşıyor mu?
Genellikle paylaşırlar — çünkü bir ilaç keşif programı tek bir kimyasal seriyi
optimize eder.

Bu gözlem Oturum 3'te önemli olacak: rastgele bir eğitim/test bölünmesi aynı
serinin moleküllerini hem eğitime hem teste dağıtır ve modelin başarısını
olduğundan iyi gösterir. Çözümü **iskelet bazlı bölünme**.

In [ ]:
def molekul_izgarasi(alt_df, baslik, satir_sayisi=6):
    moller, etiketler = [], []
    for _, satir in alt_df.head(satir_sayisi).iterrows():
        mol = Chem.MolFromSmiles(satir["Smiles"])
        if mol is None:
            continue
        moller.append(mol)
        nm = 10 ** (9 - satir[HEDEF_KOLON])
        etiketler.append(f"{HEDEF_KOLON} {satir[HEDEF_KOLON]:.2f}  ({nm:,.1f} nM)")
    print(f"\n{baslik}")
    return Draw.MolsToGridImage(moller, legends=etiketler, molsPerRow=3,
                                subImgSize=(300, 230))


molekul_izgarasi(df, "EN POTENT 6 MOLEKUL")

In [ ]:
molekul_izgarasi(df.iloc[::-1], "EN ZAYIF 6 MOLEKUL")

---
## 11. Dağılıma bakmak

Sınıflandırma eşiğini seçmeden önce dağılımı **görmemiz** gerekir.

Grafikte neye bakacağız?

- **Dağılımın merkezi ve yayılımı.** Veriler nerede yoğunlaşıyor?
- **Biçim.** Tek tepeli mi, çift tepeli mi? Çift tepe, veri setinde farklı
  kimyasal serilerin ya da farklı deney koşullarının bulunduğuna işaret eder.
- **Uçlar.** Verinin seyrek olduğu bölgelerde modelin daha zayıf tahmin yapması
  beklenir.

In [ ]:
ortalama = df[HEDEF_KOLON].mean()
medyan   = df[HEDEF_KOLON].median()
std      = df[HEDEF_KOLON].std()

fig, eksen = plt.subplots(figsize=(11, 4.6))

eksen.hist(df[HEDEF_KOLON], bins=40, color="#4878a8", edgecolor="white", alpha=0.9)
eksen.axvline(medyan, color="#c0392b", linestyle="--", linewidth=2,
              label=f"Medyan = {medyan:.2f}")
eksen.axvline(ortalama, color="#2c3e50", linestyle=":", linewidth=2,
              label=f"Ortalama = {ortalama:.2f}")
eksen.set_xlabel(HEDEF_KOLON)
eksen.set_ylabel("Molekul sayisi")
eksen.set_title(f"{TARGET_ADI} — {HEDEF_KOLON} dagilimi  (n = {len(df)}, "
                f"std = {std:.2f})")
eksen.legend()
eksen.grid(alpha=0.3, linestyle="--")

plt.tight_layout()
plt.savefig(GORSEL / "03_pic50_dagilimi.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Ortalama {ortalama:.3f}   Medyan {medyan:.3f}   Std sapma {std:.3f}")

---
## 12. Hangi eşik? Önce seçenekleri görelim

Sınıflandırma modeli için molekülleri **aktif** ve **inaktif** diye ikiye
ayırmamız gerekir. Eşik keyfi bir sayı değildir; iki şeyi birlikte düşünürüz:

**1. Farmakolojik anlam.** Kinaz inhibitörü geliştirmede `pIC50 = 7`
(IC50 ≤ 100 nM) yaygın bir "ilaç benzeri potens" sınırıdır. Daha gevşek bir
eşik (6, yani 1 µM) zayıf bağlanıcıları da aktif sayar; daha katı bir eşik
(8, yani 10 nM) yalnızca optimize edilmiş bileşikleri alır.

**2. Sınıf dengesi.** Çok dengesiz bir ayrım (ör. %95 aktif) modelin "her şeye
aktif" demesini ödüllendirir: doğruluk yüksek görünür ama model hiçbir şey
öğrenmemiştir.

Aşağıdaki tablo her eşiğin veri setimizi nasıl böldüğünü gösterir.

In [ ]:
satirlar = []
for e in np.arange(5.0, 9.01, 0.5):
    aktif = int((df[HEDEF_KOLON] >= e).sum())
    satirlar.append({
        f"Esik ({HEDEF_KOLON})": round(float(e), 2),
        f"{AKTIVITE_TIPI} (nM)": f"{10 ** (9 - e):,.0f}",
        "Aktif": aktif,
        "Inaktif": len(df) - aktif,
        "Aktif %": round(100 * aktif / len(df), 1),
    })

df_esikler = pd.DataFrame(satirlar)
df_esikler["Dengeden sapma"] = (df_esikler["Aktif %"] - 50).abs().round(1)

print("Tek esik secenekleri")
print("=" * 72)
print(df_esikler.to_string(index=False))

en_dengeli = df_esikler.loc[df_esikler["Dengeden sapma"].idxmin()]
print()
print(f"Dengeye en yakin yuvarlak esik : {en_dengeli.iloc[0]}  "
      f"(aktif %{en_dengeli['Aktif %']})")
print(f"Veriyi tam ortadan bolen deger : {medyan:.2f}  (medyan)")
print(f"Farmakolojik referans          : 7.00  (IC50 = 100 nM)")

---
## 13. Kararı verin: tek eşik mi, çift eşik mi?

İki seçenek var.

### Tek eşik

Bir sayı seçersiniz, üstü aktif, altı inaktif. Bütün moleküller kullanılır.

```
            inaktif (0)    │    aktif (1)
        ────────────────────┼────────────────────►  pIC50
                          eşik
```

**Sorun:** eşiğin hemen altındaki ve hemen üstündeki moleküller birbirine çok
benzer ama zıt etiket alır. Deneysel ölçüm hatası 0,3–0,5 pIC50 birimi
olabilirken, eşiğin 0,05 altındaki bir molekülü "inaktif" saymak modele
öğrenilmesi imkânsız bir ayrım dayatır.

### Çift eşik

Eşiğin çevresinde bir **belirsiz bant** bırakır, o banttaki molekülleri
sınıflandırma veri setinden **çıkarırsınız**.

```
     inaktif (0)   │  çıkarılan bant  │   aktif (1)
 ──────────────────┼──────────────────┼──────────────────►  pIC50
              eşik−bant           eşik+bant
```

**Kazanç:** iki sınıf net biçimde ayrışır, model daha temiz bir sınır öğrenir.
**Bedel:** veri kaybedersiniz ve test sonuçları gerçek hayattan daha iyimser
görünür — çünkü en zor örnekleri attınız. Bunu **bildirmek zorunludur**.

> Regresyon veri seti bundan etkilenmez: bant yalnızca sınıflandırmadan
> çıkarılır, `pIC50` tahmini için bütün moleküller kullanılmaya devam eder.

Aşağıdaki hücrede kararınızı verip çalıştırın.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  KARAR:  esigi burada secin ve hucreyi calistirin
# ══════════════════════════════════════════════════════════════════════════

ESIK_MODU = "tek"     # "tek"  -> tek esik, butun molekuller kullanilir
                      # "cift" -> iki esik, aradaki bant cikarilir

ESIK      = None      # None -> dagilimin medyani kullanilir (en dengeli ayrim)
                      # ya da elle bir sayi yazin: 7.0, 7.5 ...

BANT      = 0.5       # yalnizca "cift" modda: ESIK - BANT ile ESIK + BANT arasi
                      # cikarilir.  Ornek: ESIK=7.5, BANT=0.5 ->
                      # 7.0 ve alti = 0,  8.0 ve ustu = 1,  arasi atilir

# ══════════════════════════════════════════════════════════════════════════

if ESIK is None:
    ESIK = round(float(medyan), 2)
    print(f"ESIK = None verildi -> dagilimin medyani kullanildi: {ESIK}")

if ESIK_MODU == "tek":
    ALT_ESIK = UST_ESIK = float(ESIK)
    df_snf = df.copy()
    df_snf["Aktiflik"] = (df_snf[HEDEF_KOLON] >= ESIK).astype(int)
    atilan_bant = 0

elif ESIK_MODU == "cift":
    ALT_ESIK, UST_ESIK = float(ESIK) - float(BANT), float(ESIK) + float(BANT)
    bantta = df[(df[HEDEF_KOLON] > ALT_ESIK) & (df[HEDEF_KOLON] < UST_ESIK)]
    atilan_bant = len(bantta)
    df_snf = df[(df[HEDEF_KOLON] <= ALT_ESIK)
                | (df[HEDEF_KOLON] >= UST_ESIK)].copy()
    df_snf["Aktiflik"] = (df_snf[HEDEF_KOLON] >= UST_ESIK).astype(int)

else:
    raise ValueError("ESIK_MODU 'tek' ya da 'cift' olmali.")

df_snf = df_snf.reset_index(drop=True)
n_aktif = int(df_snf["Aktiflik"].sum())
n_inaktif = len(df_snf) - n_aktif

print()
print("SECILEN ESIK")
print("=" * 58)
print(f"  Mod                   : {ESIK_MODU}")
if ESIK_MODU == "tek":
    print(f"  Esik                  : {HEDEF_KOLON} >= {ESIK}"
          f"   (IC50 <= {10 ** (9 - ESIK):,.1f} nM)")
else:
    print(f"  Inaktif (0)           : {HEDEF_KOLON} <= {ALT_ESIK}")
    print(f"  Cikarilan bant        : {ALT_ESIK} < {HEDEF_KOLON} < {UST_ESIK}"
          f"   -> {atilan_bant} molekul")
    print(f"  Aktif (1)             : {HEDEF_KOLON} >= {UST_ESIK}")
print()
print(f"  Regresyon veri seti   : {len(df):>5} molekul  (bant bundan cikarilmaz)")
print(f"  Siniflandirma seti    : {len(df_snf):>5} molekul")
print(f"    Aktif (1)           : {n_aktif:>5}  (%{100 * n_aktif / len(df_snf):.1f})")
print(f"    Inaktif (0)         : {n_inaktif:>5}  (%{100 * n_inaktif / len(df_snf):.1f})")
print(f"  Dengesizlik orani     : "
      f"{max(n_aktif, n_inaktif) / max(min(n_aktif, n_inaktif), 1):.2f} : 1")

if min(n_aktif, n_inaktif) / len(df_snf) < 0.2:
    print("\n  !! Siniflar oldukca dengesiz. Oturum 4'te dengeli dogruluk ve")
    print("     MCC olcutlerine bakmayi unutmayin; duz dogruluk yaniltir.")

### Kararı grafikle görmek

Solda seçtiğiniz eşik(ler) dağılımın üzerinde. Çift eşik seçtiyseniz çıkarılan
bant gri gölgeyle işaretli. Sağda ortaya çıkan sınıf dengesi.

In [ ]:
fig, (sol, sag) = plt.subplots(1, 2, figsize=(13, 4.6),
                               gridspec_kw={"width_ratios": [1.7, 1]})

sol.hist(df[HEDEF_KOLON], bins=40, color="#bdc3c7", edgecolor="white",
         alpha=0.9, label=f"Butun veri (n={len(df)})")

if ESIK_MODU == "cift":
    sol.axvspan(ALT_ESIK, UST_ESIK, color="#7f8c8d", alpha=0.25,
                label=f"Cikarilan bant ({atilan_bant} molekul)")
    sol.axvline(ALT_ESIK, color="#c0392b", linestyle="--", linewidth=2,
                label=f"Inaktif siniri = {ALT_ESIK}")
    sol.axvline(UST_ESIK, color="#27ae60", linestyle="--", linewidth=2,
                label=f"Aktif siniri = {UST_ESIK}")
else:
    sol.axvline(ESIK, color="#c0392b", linestyle="--", linewidth=2.2,
                label=f"Esik = {ESIK}")

sol.set_xlabel(HEDEF_KOLON)
sol.set_ylabel("Molekul sayisi")
sol.set_title(f"Secilen esik — mod: {ESIK_MODU}")
sol.legend(fontsize=8.5)
sol.grid(alpha=0.3, linestyle="--")

etiket = [f"Inaktif\n(0)", f"Aktif\n(1)"]
cubuk = sag.bar(etiket, [n_inaktif, n_aktif],
                color=["#95a5a6", "#27ae60"], edgecolor="white")
for c, v in zip(cubuk, [n_inaktif, n_aktif]):
    sag.text(c.get_x() + c.get_width() / 2, v,
             f"{v}\n(%{100 * v / len(df_snf):.1f})",
             ha="center", va="bottom", fontsize=10)
sag.set_ylabel("Molekul sayisi")
sag.set_title(f"Sinif dengesi (n = {len(df_snf)})")
sag.set_ylim(0, max(n_aktif, n_inaktif) * 1.22)
sag.grid(alpha=0.3, axis="y", linestyle="--")

plt.tight_layout()
plt.savefig(GORSEL / "04_esik_ve_sinif_dengesi.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 14. Veri setlerini kaydetmek

Üç dosya yazıyoruz:

| Dosya | İçeriği | Kullanan |
|---|---|---|
| `ntrk1_regresyon.csv` | `Smiles` + `pIC50` | Oturum 3 |
| `ntrk1_siniflandirma.csv` | `Smiles` + `pIC50` + `Aktiflik` | Oturum 4 |
| `esik_secimi.json` | eşik kararınız | Oturum 2, 4, 5, 6 |

`esik_secimi.json` önemli: sonraki defterler eşiği **yeniden seçmez**, bu
dosyadan okur. Böylece bir kez karar verir, bütün kurs boyunca aynı eşikle
çalışırsınız.

Yanına bir de **künye** yazıyoruz: hangi kaynaktan, hangi tarihte, kaç
molekülle üretildiği. Aylar sonra "bu dosya neydi?" sorusunu yanıtlamanın en
kolay yolu.

In [ ]:
from datetime import datetime

# Regresyon: butun molekuller
df[["Smiles", HEDEF_KOLON]].to_csv(YOL_REG, index=False)

# Siniflandirma: cift esik secildiyse bant cikarilmis hali
df_snf[["Smiles", HEDEF_KOLON, "Aktiflik"]].to_csv(YOL_SNF, index=False)

# Esik karari — sonraki oturumlar bunu okuyacak
esik_secimi = {
    "mod": ESIK_MODU,
    "esik": float(ESIK),
    "bant": float(BANT) if ESIK_MODU == "cift" else 0.0,
    "alt_esik": float(ALT_ESIK),
    "ust_esik": float(UST_ESIK),
    "hedef_kolon": HEDEF_KOLON,
    "cikarilan_bant_molekul": int(atilan_bant),
    "regresyon_molekul": int(len(df)),
    "siniflandirma_molekul": int(len(df_snf)),
    "aktif": int(n_aktif),
    "inaktif": int(n_inaktif),
    "aktif_yuzde": round(100 * n_aktif / len(df_snf), 2),
}
yol_esik = BU / "esik_secimi.json"
yol_esik.write_text(json.dumps(esik_secimi, ensure_ascii=False, indent=2),
                    encoding="utf-8")

# Kunye
kunye = {
    "hedef": TARGET_ADI,
    "hedef_chembl_id": TARGET_CHEMBL_ID,
    "uniprot": "P04629",
    "aktivite_tipi": AKTIVITE_TIPI,
    "hedef_kolon": HEDEF_KOLON,
    "veri_kaynagi": HAM_KAYNAK,
    "uretim_zamani": datetime.now().isoformat(timespec="seconds"),
    "mantik_araligi": list(MANTIK_ARALIGI),
    "huni": {etiket: int(n) for etiket, n in huni},
    "esik_secimi": esik_secimi,
    f"{HEDEF_KOLON}_ortalama": round(float(df[HEDEF_KOLON].mean()), 3),
    f"{HEDEF_KOLON}_medyan": round(float(df[HEDEF_KOLON].median()), 3),
    f"{HEDEF_KOLON}_std": round(float(df[HEDEF_KOLON].std()), 3),
}
yol_kunye = BU / "ntrk1_oturum01_kunye.json"
yol_kunye.write_text(json.dumps(kunye, ensure_ascii=False, indent=2),
                     encoding="utf-8")

print("Kaydedilen dosyalar")
print("-" * 62)
for yol in (YOL_HAM, YOL_REG, YOL_SNF, yol_esik, yol_kunye):
    print(f"  {yol.name:<34} {yol.stat().st_size / 1024:>9.1f} KB")
print(f"\nKlasor: {BU}")
print(f"Grafikler: {GORSEL}")

---
## 15. Özet

### Ne yaptık?

1. ChEMBL'e dayanıklı (yeniden denemeli) sorgu atan bir arayüz yazdık.
2. Hedefteki bütün ölçüm tiplerini saydık ve en çok veriye sahip olanı seçtik.
3. Ham IC50 kayıtlarını **dokuz adımda** temizledik; her adımda ne elendiğini
   gördük.
4. `pIC50` hedefini hesapladık ve ChEMBL'in kendi değeriyle doğruladık.
5. Dağılıma bakıp sınıflandırma eşiğini **tek ya da çift** olarak seçtik.
6. İki veri setini ve eşik kararını Drive'a kaydettik.

### Akılda kalması gerekenler

- **Veri hazırlığı modelleme kadar önemlidir.** Sızıntılı ya da tutarsız bir
  veri seti, en iyi algoritmayı da işe yaramaz hâle getirir.
- **Kanonikleştirme zorunluluktur.** Gizli yapısal tekrarlar eğitim/test
  bölünmesini bozar ve modeli olduğundan iyi gösterir.
- **Sansürlü değerleri (`>`, `<`) regresyona sokmayın.**
- **Birim tutarlılığı** olmadan pIC50 anlamsızdır.
- **Eşik bilimsel bir karardır:** farmakolojik anlam önce, sınıf dengesi sonra.
  Ne seçtiyseniz bildirin.
- **Veri sürümünü bildirin:** ChEMBL sürümü + indirme tarihi + filtreler.

### Sırada ne var?

**Oturum 2 — Özellik üretimi.** `Smiles` kolonu hâlâ bir **metin**; model ise
**sayılarla** öğrenir. Bir sonraki defterde her molekülü iki tamamlayıcı yolla
sayıya çevireceğiz: yorumlanabilir **moleküler betimleyiciler** ve yapısal
**parmak izleri**.

---

### Alıştırmalar

1. Eşiği `"cift"` moda alıp `BANT = 0.5` ile çalıştırın. Kaç molekül
   çıkarıldı, sınıf dengesi nasıl değişti?
2. `TARGET_CHEMBL_ID`'yi `CHEMBL2599` (SYK kinazı) ya da `CHEMBL267`
   (SRC kinazı) yapıp defteri baştan çalıştırın. Temizlik hunisi nasıl
   değişiyor? Hangi hedefin verisi daha temiz?
3. Adım 5'te molekül başına **minimum** değeri seçtik. **Medyan** almak daha mı
   iyi olurdu? Hangi durumda hangisi tercih edilir?
4. Adım 1'de attığımız sansürlü kayıtlar kaç taneydi? Bunları modele katmanın
   bir yolu var mı? (İpucu: *censored regression*)